# 构建外部工具箱

定义外部工具箱类

In [22]:
class AgentToolkit():
    def __init__(self):
        self.tools = {}
        
    def add(self, tool_func, tool_desp): 
        """
        添加工具。
        Args:
            tool_func: 实现该工具的函数对象
            tool_desp: 关于该工具的描述
        Returns:
            None: 无返回值
        """
        self.tools[tool_func.__name__] = {"func": tool_func, "info": tool_desp}
    
    def get(self, tool_name): 
        """
        调用工具
        """
        return self.tools.get(tool_name, {}).get("func", None)

    def describe(self): 
        """
        获取所有工具的描述
        """
        desp = [f"{tool_name}: {tool['info']}" for tool_name,tool in self.tools.items()]
        return "\n".join(desp)

基于akshare的股价获取工具

In [23]:
from datetime import date, timedelta
import akshare as ak

def get_yesterday_stock(stock_id):
    target_date = date.today() - timedelta(days=1) # 以调用时刻反推昨日
    target_date = target_date.strftime("%Y%m%d")
    df = ak.stock_zh_a_hist(symbol=stock_id, period='daily', start_date=target_date, end_date=target_date, adjust='qfq')
    record = df.loc[0].to_dict()
    record['日期'] = record['日期'].strftime("%Y%m%d")
    record = [f"{key}{value}" for key, value in record.items()]
    record = '，'.join(record) + '。'
    return record


基于腾讯财经的股价获取工具

In [24]:
from datetime import date, timedelta
import requests

TIMEOUT = 10

def get_market_prefix(stock_id):
    """由 6 位股票代码推断交易所前缀：sh（上交所）/ sz（深交所）/ bj（北交所）。"""
    if stock_id.startswith(('60', '68', '90', '11', '13')):
        return 'sh'
    if stock_id.startswith(('00', '30', '20', '12')):
        return 'sz'
    return 'bj'

def format_record(record):
    return '，'.join(f'{key}{value}' for key, value in record.items()) + '。'

def get_yesterday_stock(stock_id):
    """请求腾讯财经 qt.gtimg.cn 快照接口，并校验其日期确为昨日。"""
    target_date = (date.today() - timedelta(days=1)).strftime('%Y%m%d')
    url = f'https://qt.gtimg.cn/q={get_market_prefix(stock_id)}{stock_id}'
    resp = requests.get(url, timeout=TIMEOUT)
    resp.encoding = 'gbk'
    fields = resp.text.strip().split('="')[1].rstrip('";').split('~')
    # 快照始终返回最新交易日，若不是昨日则视为取数失败
    fields = [fields] if fields[30][:8] == target_date else []
    fields = fields[0]                   # 昨日非交易日时列表为空，这里会报错
    record = {
        '日期': target_date,
        '开盘': float(fields[5]),
        '收盘': float(fields[3]),
        '最高': float(fields[33]),
        '最低': float(fields[34]),
        '成交量': float(fields[36]),
        '成交额': float(fields[37]) * 10000,     # 原单位为万元
    }
    return format_record(record)


邮件发送工具

In [25]:
import smtplib
from email.mime.text import MIMEText

def send_email(subject: str, content: str, receiver: str):
    sender = "xxxx@xxx.com"  # 发件箱
    authcode = "xxxx"  # 发件箱授权码（注意不是登陆密码），在邮箱SMTP设置页面查看
    smtp_host = "xxxx"  # 发件箱SMTP服务器地址（新浪为smtp.sina.com，163为smtp.163.com，qq为smtp.qq.com）
    msg = MIMEText(content, "plain", "utf-8")  # 创建邮件对象
    msg["Subject"], msg["From"], msg["To"] = subject, sender, receiver
    try:
        with smtplib.SMTP_SSL(smtp_host, 465) as server:
            server.login(sender, authcode)
            server.send_message(msg)
        return "邮件已发送"
    except Exception as e:
        return f"邮件发送失败: {e}"

# 工具绑定

In [26]:
toolkit = AgentToolkit()
toolkit.add(get_yesterday_stock, "获取指定股票的昨日交易数据，参数stock_id表示股票代码")
toolkit.add(send_email, "发送电子邮件，输入邮件主题和内容，发送邮件到指定收件人。参数subject表示邮件主题，content表示邮件内容，receiver表示收件人邮箱地址")
print(toolkit.describe())


get_yesterday_stock: 获取指定股票的昨日交易数据，参数stock_id表示股票代码
send_email: 发送电子邮件，输入邮件主题和内容，发送邮件到指定收件人。参数subject表示邮件主题，content表示邮件内容，receiver表示收件人邮箱地址


# 提示词构建

In [27]:
def apply_chat_template(prompt, toolkit_desp, previous_state=''):
    prompt_template = f"""
    你是一个智能助理，能够调用外部工具来完成相关的任务。
    可以使用的工具如下：
    {toolkit_desp}

    你的回答必须是以下两种形式之一，且不能同时进行：
    1、如果你需要调用工具来获取信息，按照“行动：[工具名称];[JSON格式，参数名:参数值]”的格式进行回答。
    2、如果你已经有足够的信息来回答用户的问题，按照“回答：[你的回答内容]”的格式进行回答。

    任务如下：
    {prompt}
    之前返回结果：
    {previous_state}
    """
    return prompt_template
# 函数调用示例
task_prompt = "查询工商银行（股票代码：601398）昨日的股票交易数据，并将数据发送到邮箱xxxx@ruc.edu.cn"
res = apply_chat_template(task_prompt, toolkit.describe())

# 智能体主体架构

In [32]:
import json
from openai import OpenAI

class LLMClient():
    def __init__(self, my_key):
        workspace_id = 'your_workspace_id'
        self.client = OpenAI(api_key=my_key, base_url=f'https://{workspace_id}.cn-beijing.maas.aliyuncs.com/compatible-mode/v1')    
    def _process (self, prompt, model="qwen3-max"):
        messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content":prompt}]
        completion = self.client.chat.completions.create(model=model, messages=messages)
        resp = completion.to_dict()['choices'][0]['message']['content']
        return resp

    def answer(self, prompt):
        return self._process(prompt)
        
class Agent:
    def __init__(self, api_key, toolkit):
        self.toolkit = toolkit
        self.llm = LLMClient(api_key)
        self.max_steps = 5
        
    def _process(self, prompt):
        previous_results = []
        tk_desp = self.toolkit.describe()
        for i in range(self.max_steps):
            full_prompt = apply_chat_template(prompt, tk_desp, '\n'.join(previous_results))
            resp = self.llm.answer(full_prompt).strip() 
            print(f"Step {i+1}. llm> {resp}")
            if resp.startswith("回答："):
                return {'status':True, 'message': resp[len("回答："):]}
            elif resp.startswith("行动："): 
                action_part = resp[len("行动："):]
                tool_name, param_str = action_part.split(';', 1)
                param_str = json.loads(param_str)
                tool_func = self.toolkit.get(tool_name)
                if tool_func:
                    res = tool_func(**param_str)
                    previous_results.append(res) # 收集外部工具的执行结果
                else:
                    return {'status':False, 'message':f"未找到工具 {tool_name}"}
            else:
                return {'status':False, 'message': "无法解析的响应格式"}
        return {'status':False, 'message': "达到最大步骤数，未能得到最终回答。"}

    def answer(self, prompt):
        resp = self._process(prompt)
        return resp['message']

# 智能体调用

In [ ]:
task_prompt = "查询工商银行（股票代码：601398）昨日的股票交易数据，并将数据发送到邮箱xxxx@ruc.edu.cn"

agent = Agent("your_own_api_key", toolkit)
resp = agent.answer(task_prompt)